# Prepare BARRA-C2 data for analysis

- Run tas average for each state greater metro region

In [27]:
import xarray as xr
import os
import geopandas as gpd
import regionmask

### User configuraiton

In [42]:
years = range(2015, 2024)

In [21]:
barra_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/"
write_path = "/home/565/ad1803/Hot_Cloudy/Demand/sanaa_work/"

In [22]:
fy = str(min(years))
ly = str(max(years))

variables = {
    "tas": [
        barra_path + "tas/latest/",  # directory containing monthly files
        "lat",  # latitude coordinate name in files
        "lon",  # longitude coordinate name in files
        write_path + "tas/tas_barra-c2_hourly_" + fy + "-" + ly + "_FILTERED",
        "tas",  # original variable name
        "tas",  # new variable name
    ]
}

month_str = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

### Load & filter region shape file

In [24]:
shp_path = "/home/565/ad1803/Hot_Cloudy/Demand/sanaa_work/shapefiles/GCCSA_2021_AUST_GDA2020.shp"
gdf = gpd.read_file(shp_path)

In [33]:
#keep only regions that match Carl's solar areas
gdf = gdf[gdf["GCC_CODE21"].isin(["1GSYD", "2GMEL", "3GBRI","4GADE","6GHOB","8ACTE"])]
#gdf

In [25]:
# Ensure geographic CRS
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

In [32]:
#rasterise shapefiles
# Build regions directly from GeoDataFrame
regions = regionmask.from_geopandas(gdf, names="GCC_NAME21", name="GCCSA21")

### Functions

In [38]:
# Open multiple hourly files and subset to lat/lon slices
def open_hourly(file_list, lat_slice, lon_slice, lat_name="lat", lon_name="lon"):

    def preprocess(ds):
        ds = ds.rename({lat_name: "lat", lon_name: "lon"})
        return ds.sel(lon=lon_slice, lat=lat_slice)

    ds = xr.open_mfdataset(
        file_list,
        preprocess=preprocess,
        chunks={"time": "200MB"},
        combine="by_coords",
        parallel=True,
    )
    return ds

In [39]:
#Compute unweighted spatial mean for each region
def region_mean(ds, mask_da, region_names):
    grouped = ds.where(mask_da.notnull()).groupby(mask_da)
    reg_mean = grouped.mean(["lat", "lon"])
    reg_mean = reg_mean.rename({mask_da.name: "region"})
    reg_mean = reg_mean.assign_coords(region=("region", region_names))
    return reg_mean

In [40]:
#Write dataset to NetCDF
def write(ds, fp):
    ds = ds.chunk({"time": -1})
    ds.to_netcdf(fp + ".nc")

### Average over regions

Dictionary of variables to process.

Organised with the variable name as the key, then a list as follows:
`[path_to_open, lat_name, lon_name, path_to_write, var_name, new_var_name]`

In [45]:
for key, values in zip(variables.keys(), variables.values()):
    for year in years:
        print(f"Processing {key} for year: {year}")

        files_to_open = [
            f"{values[0]}{key}_AUST-04_ERA5_historical_hres_BOM_BARRA-C2_v1_1hr_{year}{m}-{year}{m}.nc"
            for m in month_str
        ]
        print(f"Opening files: {os.path.basename(files_to_open[0])} ...")

        ds = open_hourly(files_to_open, slice(-44, -10), slice(125, 154), values[1], values[2])

        # Create raster mask for filtered regions
        mask = regions.mask(ds)
        mask.name = "region_id"

        # Compute regional means
        reg_ds = region_mean(ds, mask, gdf["GCC_NAME21"].tolist())

        # Rename variable
        if values[4] in reg_ds:
            reg_ds = reg_ds.rename({values[4]: values[5]})

        fp = values[3] + f"_{year}"
        print(f"Writing to: {fp}.nc")
        write(reg_ds, fp)

print("All done.")

Processing tas for year: 2015
Opening files: tas_AUST-04_ERA5_historical_hres_BOM_BARRA-C2_v1_1hr_201501-201501.nc ...


AttributeError: module 'dask' has no attribute 'utils'

# Close cluster